# Revised compact: report-informed, trial-safe experiments

Uses the original archive and frozen seed-42 folds. All 945 validation windows are primary; the historical 939-window population is exported separately. Trial-pooled predictions are a separate endpoint. Existing results are preserved in the earlier notebooks.

New features are **report-inspired**, not a reproduction of the report's 36/51-feature pipelines. Shape10 subtracts the mean log band power separately in each ear. Temporal12 measures the slope and spread of five one-second log variances in theta, alpha and beta, separately in each ear. Both are invariant to a constant positive gain in either ear, away from numerical floors. This does not remove muscle artifacts.

The MoE uses three balanced logistic-regression experts (C=.1). Quality routing uses amplitude, crest factor, energy concentration and high-frequency fraction. These are recording descriptors, not validated artifact labels. Inner predictions are pooled by trial before learning the gate. The gate falls back to equal weights when any class has fewer than two training trials. No held-out labels train the gate.

Default is seed 42, all subjects and both training inclusion policies. Change SEEDS only before running; suggested robustness run [42,7,21,84,2026]. These reused data support exploratory comparisons, not an independent confirmation of a selected winner. See REVIEW_AND_ABLATIONS.md for controls and limitations.


In [ ]:
import importlib.util, subprocess, sys
packages={'numpy':'numpy','scipy':'scipy','pandas':'pandas','sklearn':'scikit-learn','matplotlib':'matplotlib','xgboost':'xgboost'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import io,json,hashlib,platform,warnings,zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import butter,sosfiltfilt,welch,resample_poly,iirnotch,filtfilt
from scipy.integrate import trapezoid
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.feature_selection import f_classif,mutual_info_classif
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,confusion_matrix,ConfusionMatrixDisplay
from xgboost import XGBClassifier
try:
    from IPython.display import display, FileLink
except ImportError:
    def display(x): print(x)
    def FileLink(x): return str(x)
VERSIONS={p:version(p) for p in packages.values()}
print(platform.python_version(),VERSIONS)
import itertools,time

RUN_STARTED=time.time()

In [ ]:
SEED=42; FS=500; CHANNELS=[0,3]; WINDOW=2500; STEP=1250
STARTS=[0,1250,2500]
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,12),'beta':(12,30),'gamma':(30,45)}
FILTER_ORDER=4; N_SELECTED=20; BALANCE=True; LABELS=np.array([-1,0,1])
SUBJECT_IDS=None
SUSPECTED_TRIALS={(1,6),(7,21)}
SIGNAL_UNIT='uV'; REMOVE_TRIAL_CONSTANT=True; ADDITIONAL_NOTCH=False
ARCHIVE_PATH=None
DRIVE_ID='1Ma1fHzvhwyQm2zbtYXV8H-SACrmGH88L'
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT=WORK/'revised_compact_results';OUT.mkdir(parents=True,exist_ok=True)
if not Path('/kaggle/working').exists() and ARCHIVE_PATH is None and (Path.cwd()/'LMB_05_HW1.zip').exists():
    ARCHIVE_PATH=str(Path.cwd()/'LMB_05_HW1.zip')
EPS=1e-30; TEMPORAL_FS=128; PERM_ORDER=3; PERM_DELAY=1
config=dict(seed=SEED,fs=FS,channels_zero_based=CHANNELS,window=WINDOW,step=STEP,bands=BANDS,
    filter_order=FILTER_ORDER,n_selected_original_control=N_SELECTED,balance=BALANCE,
    suspected_trials=sorted(SUSPECTED_TRIALS),signal_unit_assumed=SIGNAL_UNIT,
    units_independently_verified=False,remove_trial_constant=REMOVE_TRIAL_CONSTANT,
    additional_notch=ADDITIONAL_NOTCH,wavelet=False,pretrained=False,
    temporal_features_fs=TEMPORAL_FS,permutation_order=PERM_ORDER,permutation_delay=PERM_DELAY,
    power_floor=EPS,versions=VERSIONS,
    primary_evaluation='same nonsuspected validation windows under both training policies',
    secondary_evaluation='all original validation windows and suspected-trial diagnostics',
    model_selection='none; fixed settings; comparison is exploratory')
print('Output:',OUT)


In [ ]:
if ARCHIVE_PATH is not None:
    archive=Path(ARCHIVE_PATH)
else:
    candidates=sorted(Path('/kaggle/input').rglob('LMB_05_HW1.zip')) if Path('/kaggle/input').exists() else []
    if len(candidates)>1: raise ValueError('Set ARCHIVE_PATH: multiple ZIPs found.')
    archive=candidates[0] if candidates else WORK/'LMB_05_HW1.zip'
if not archive.exists():
    if importlib.util.find_spec('gdown') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','gdown'])
    import gdown
    gdown.download(id=DRIVE_ID,output=str(archive),quiet=False)
assert zipfile.is_zipfile(archive), 'Expected ZIP, not an HTML access error.'
config['archive_sha256']=hashlib.sha256(archive.read_bytes()).hexdigest()
subjects={}
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    for name in [n for n in names if n.endswith('.emotion_preprocessed_epoch_InVideo_new.mat')]:
        folder=name.rsplit('/',1)[0]; sid=int(folder.rsplit('/',1)[-1])
        label_files=[n for n in names if n.startswith(folder+'/') and n.endswith('labels.mat')]
        assert len(label_files)==1,(sid,label_files)
        cell=loadmat(io.BytesIO(z.read(name)))['Preprocessed_trials']
        y=loadmat(io.BytesIO(z.read(label_files[0])))['matching_ratings'].ravel()
        assert cell.shape==(1,21) and y.shape==(21,) and np.isin(y,LABELS).all()
        trials=np.stack([np.asarray(t,dtype=np.float64) for t in cell.ravel()])
        assert trials.shape==(21,6,5000),sid
        assert np.isfinite(trials[:,CHANNELS,:]).all(),f'Selected ear contains nonfinite data: {sid}'
        assert sid not in subjects
        subjects[sid]={'trials':trials[:,CHANNELS,:],'y':y.astype(int), 'unused_nonfinite':int((~np.isfinite(trials)).sum())}
assert sorted(subjects)==[1,3,5,7,9,11,13,15,17,19,21,23,27,29,31]
subjects=dict(sorted(subjects.items()))
inventory=pd.DataFrame([dict(subject=sid,trials=21,unused_channel_nonfinite=d['unused_nonfinite'],**{f'label_{c}':int((d['y']==c).sum()) for c in LABELS}) for sid,d in subjects.items()])
display(inventory)
print('Archive SHA-256:',config['archive_sha256'])

assert SIGNAL_UNIT in ['V','mV','uV']
nb,na=iirnotch(50,30,fs=FS)
for sid,d in subjects.items():
    d['original_selected_ears']=d['trials'].copy()
    d['original_trial_numbers']=np.arange(1,22)
    if REMOVE_TRIAL_CONSTANT:d['trials']=d['trials']-d['trials'].mean(axis=-1,keepdims=True)
    if ADDITIONAL_NOTCH:d['trials']=filtfilt(nb,na,d['trials'],axis=-1)
    d['segments']=np.stack([trial[:,start:start+2500] for trial in d['trials'] for start in [0,1250,2500]])
    d['segment_y']=np.repeat(d['y'],3)
    d['trial_ids']=np.repeat(d['original_trial_numbers'],3)
    assert d['segments'].shape==(63,2,2500) and np.isfinite(d['segments']).all()
active_ids=sorted(subjects) if SUBJECT_IDS is None else list(SUBJECT_IDS)
assert active_ids and set(active_ids)<=set(subjects)
assert SUSPECTED_TRIALS <= {(sid,t) for sid in subjects for t in range(1,22)}
print('Original data retained:',len(subjects),'subjects,',sum(len(d['y']) for d in subjects.values()),'trials.',flush=True)

In [ ]:
FROZEN_FOLD_IDS={1: [{'fold': 1, 'train_trials': [2, 3, 4, 5, 8, 11, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 6, 7, 9, 10, 12, 19]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 18, 19], 'validation_trials': [2, 14, 15, 16, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 5, 8, 11, 13, 18]}], 3: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 13, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 13, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 14, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 17, 21], 'validation_trials': [3, 7, 15, 16, 18, 19, 20]}], 5: [{'fold': 1, 'train_trials': [2, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 3, 4, 5, 7, 12, 21]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 11, 12, 13, 15, 18, 19, 20, 21], 'validation_trials': [2, 8, 9, 10, 14, 16, 17]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 14, 16, 17, 21], 'validation_trials': [6, 11, 13, 15, 18, 19, 20]}], 7: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 13, 14, 15, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 16, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 14, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 13, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 16, 17, 21], 'validation_trials': [3, 7, 14, 15, 18, 19, 20]}], 9: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 8, 10, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 9, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [7, 8, 14, 15, 16, 18, 20]}], 11: [{'fold': 1, 'train_trials': [2, 3, 4, 6, 8, 9, 10, 12, 15, 17, 18, 19, 20, 21], 'validation_trials': [1, 5, 7, 11, 13, 14, 16]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 18], 'validation_trials': [2, 8, 12, 17, 19, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 7, 8, 11, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 6, 9, 10, 15, 18]}], 13: [{'fold': 1, 'train_trials': [3, 4, 6, 7, 9, 10, 12, 13, 14, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 5, 8, 11, 15, 16]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 11, 13, 14, 15, 16, 18, 19], 'validation_trials': [6, 9, 10, 12, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 6, 8, 9, 10, 11, 12, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 7, 13, 14, 18, 19]}], 15: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 10, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 9, 10, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [6, 7, 14, 15, 16, 18, 20]}], 17: [{'fold': 1, 'train_trials': [1, 3, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 9, 10, 11, 15, 18]}, {'fold': 2, 'train_trials': [2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 18, 21], 'validation_trials': [1, 5, 13, 16, 17, 19, 20]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 9, 10, 11, 13, 15, 16, 17, 18, 19, 20], 'validation_trials': [3, 6, 7, 8, 12, 14, 21]}], 19: [{'fold': 1, 'train_trials': [3, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 6, 11, 13]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 7, 9, 11, 13, 14, 16, 18, 20], 'validation_trials': [8, 10, 12, 15, 17, 19, 21]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 10, 11, 12, 13, 15, 17, 19, 21], 'validation_trials': [3, 7, 9, 14, 16, 18, 20]}], 21: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 16, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 10, 12, 13, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 8, 9, 11, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 16, 17, 19, 21], 'validation_trials': [7, 12, 13, 14, 15, 18, 20]}], 23: [{'fold': 1, 'train_trials': [1, 3, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 5, 8, 11, 13, 18]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 8, 9, 11, 12, 13, 14, 15, 16, 18, 20], 'validation_trials': [3, 6, 7, 10, 17, 19, 21]}, {'fold': 3, 'train_trials': [2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 17, 18, 19, 21], 'validation_trials': [1, 9, 12, 14, 15, 16, 20]}], 27: [{'fold': 1, 'train_trials': [3, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 6, 7, 8, 9]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 9, 10, 16, 17, 18, 19, 21], 'validation_trials': [3, 11, 12, 13, 14, 15, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 20], 'validation_trials': [5, 10, 16, 17, 18, 19, 21]}], 29: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 10, 11]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 10, 11, 14, 16, 18, 19, 21], 'validation_trials': [3, 9, 12, 13, 15, 17, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 15, 17, 20], 'validation_trials': [6, 7, 14, 16, 18, 19, 21]}], 31: [{'fold': 1, 'train_trials': [1, 2, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 8, 11, 13, 18, 19]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 18, 19, 21], 'validation_trials': [7, 9, 14, 15, 16, 17, 20]}, {'fold': 3, 'train_trials': [3, 4, 7, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 5, 6, 10, 12, 21]}]}
ORIGINAL_FOLD_FILE_SHA256='d657d62bc72561bf7c6abaffcd9df319eab91f61ce129d665ba101ed4adfebc4'

config['original_fold_file_sha256']=ORIGINAL_FOLD_FILE_SHA256
folds={};split_rows=[]
for sid in active_ids:
    d=subjects[sid];seen=[];folds[sid]=[]
    for item in FROZEN_FOLD_IDS[sid]:
        train_ids=np.asarray(item['train_trials'],int);val_ids=np.asarray(item['validation_trials'],int)
        assert not set(train_ids)&set(val_ids)
        assert set(train_ids)|set(val_ids)==set(range(1,22))
        seen.extend(val_ids.tolist())
        suspected_ids={t for s,t in SUSPECTED_TRIALS if s==sid}
        train=np.flatnonzero(np.isin(d['trial_ids'],train_ids))
        train_without=np.flatnonzero(np.isin(d['trial_ids'],[t for t in train_ids if t not in suspected_ids]))
        val_all=np.flatnonzero(np.isin(d['trial_ids'],val_ids))
        val_common=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t not in suspected_ids]))
        val_suspect=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t in suspected_ids]))
        assert len(train)==42 and len(val_all)==21 and len(val_common)>0
        assert set(val_all)==set(val_common)|set(val_suspect)
        assert not set(val_common)&set(val_suspect)
        assert not set(d['trial_ids'][train_without])&suspected_ids
        fold=dict(fold=item['fold'],train_include=train,train_exclude=train_without,
                  val_all=val_all,val_common=val_common,val_suspect=val_suspect)
        folds[sid].append(fold)
        split_rows.append(dict(subject=sid,fold=item['fold'],
            train_include_trials=json.dumps(train_ids.tolist()),
            train_exclude_trials=json.dumps(np.unique(d['trial_ids'][train_without]).tolist()),
            common_validation_trials=json.dumps(np.unique(d['trial_ids'][val_common]).tolist()),
            suspected_validation_trials=json.dumps(np.unique(d['trial_ids'][val_suspect]).tolist()),
            include_training_windows=len(train),exclude_training_windows=len(train_without),
            common_validation_windows=len(val_common),suspected_validation_windows=len(val_suspect),
            training_data_changed=not np.array_equal(train,train_without),
            **{f'include_train_label_{c}':int((d['segment_y'][train]==c).sum()) for c in LABELS},
            **{f'exclude_train_label_{c}':int((d['segment_y'][train_without]==c).sum()) for c in LABELS},
            **{f'common_val_label_{c}':int((d['segment_y'][val_common]==c).sum()) for c in LABELS}))
    assert sorted(seen)==list(range(1,22))
split_audit=pd.DataFrame(split_rows);split_audit.to_csv(OUT/'fixed_fold_comparison_audit.csv',index=False)
display(split_audit[split_audit.subject.isin([s for s,t in SUSPECTED_TRIALS])])
print('Folds frozen. No split regeneration.',flush=True)
class_rows=[]
for sid in active_ids:
    d=subjects[sid]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',UserWarning)
        regenerated=list(StratifiedKFold(3,shuffle=True,random_state=SEED).split(np.zeros(len(d['y'])),d['y']))
    for frozen,(tr,va) in zip(FROZEN_FOLD_IDS[sid],regenerated):
        assert np.array_equal(np.sort(tr+1),np.sort(frozen['train_trials']))
        assert np.array_equal(np.sort(va+1),np.sort(frozen['validation_trials']))
        for label in LABELS:
            class_rows.append(dict(subject=sid,fold=frozen['fold'],label=int(label),
                total_trials=int((d['y']==label).sum()),training_trials=int((d['y'][tr]==label).sum()),
                validation_trials=int((d['y'][va]==label).sum())))
class_audit=pd.DataFrame(class_rows);class_audit.to_csv(OUT/'stratification_audit.csv',index=False)
display(class_audit[class_audit.total_trials<3])
print('PASS: original folds exactly reproduced from trial-level stratification.')


In [ ]:
STARTS=[0,1250,2500]
STAT_NAMES=['mean','std','diff1','norm_diff1','diff2','norm_diff2','relative_power','DE']
FEATURE_NAMES=[f'{ear}_{band}_{stat}' for ear in ['right','left'] for band in BANDS for stat in STAT_NAMES]
SOS={b:butter(FILTER_ORDER,limits,btype='bandpass',fs=FS,output='sos') for b,limits in BANDS.items()}
def integrate_psd(f,p,lo,hi):
    inside=(f>lo)&(f<hi)
    grid=np.r_[lo,f[inside],hi]
    return trapezoid(np.r_[np.interp(lo,f,p),p[inside],np.interp(hi,f,p)],grid)
def handcrafted_features(trials):
    rows=[]
    for trial in trials:
        filtered={b:sosfiltfilt(sos,trial,axis=-1) for b,sos in SOS.items()}
        for start in STARTS:
            row=[]
            for ear in range(2):
                raw=trial[ear,start:start+WINDOW]
                f,p=welch(raw,fs=FS,window='hann',nperseg=1000,noverlap=500,detrend='constant',scaling='density')
                den=integrate_psd(f,p,1,100)
                for band,(lo,hi) in BANDS.items():
                    x=filtered[band][ear,start:start+WINDOW]
                    mu,sd=x.mean(),x.std(ddof=0)
                    z=(x-mu)/sd if sd>0 else np.zeros_like(x)
                    rp=integrate_psd(f,p,lo,hi)/den if den>0 else 0.0
                    row.extend([mu,sd,np.abs(np.diff(x)).mean(),np.abs(np.diff(z)).mean(),
                                np.abs(x[2:]-x[:-2]).mean(),np.abs(z[2:]-z[:-2]).mean(),rp,
                                .5*np.log(2*np.pi*np.e*max(sd**2,1e-30))])
            rows.append(row)
    X=np.asarray(rows)
    assert X.shape==(len(trials)*3,80) and np.isfinite(X).all()
    return X

EARS=['right','left']
A_NAMES=[f'{ear}_{band}_log_power' for ear in EARS for band in BANDS]
B_NAMES=A_NAMES+[f'left_minus_right_{band}_log_power' for band in BANDS]+[
    f'{ear}_{ratio}_log_ratio' for ear in EARS for ratio in ['beta_over_alpha','theta_over_alpha']]
C_NAMES=B_NAMES+[f'{ear}_{feature}' for ear in EARS for feature in
    ['hjorth_mobility','hjorth_complexity','spectral_entropy','permutation_entropy']]
FEATURE_SETS={'A10':A_NAMES,'B19':B_NAMES,'C27':C_NAMES,'Original80':FEATURE_NAMES}
assert [len(FEATURE_SETS[k]) for k in ['A10','B19','C27','Original80']]==[10,19,27,80]
TEMPORAL_SOS=butter(4,[1,45],btype='bandpass',fs=FS,output='sos')
PATTERNS=list(itertools.permutations(range(PERM_ORDER)))

def hjorth(x):
    x=np.asarray(x,float);dx=np.diff(x)*TEMPORAL_FS;ddx=np.diff(dx)*TEMPORAL_FS
    vx,vd,vdd=np.var(x),np.var(dx),np.var(ddx)
    if vx<=EPS:return 0.,0.
    mobility=np.sqrt(vd/vx)
    complexity=np.sqrt(vdd/vd)/mobility if vd>EPS and mobility>0 else 0.
    return float(mobility),float(complexity)

def spectral_entropy(f,p):
    v=np.asarray(p[(f>=1)&(f<=45)],float)
    if v.sum()<=EPS:return 0.
    probabilities=v/v.sum();positive=probabilities[probabilities>0]
    return float(-np.sum(positive*np.log(positive))/np.log(len(v)))

def permutation_entropy(x):
    n=len(x)-(PERM_ORDER-1)*PERM_DELAY
    if n<=0:raise ValueError('Insufficient samples for ordinal patterns')
    embedded=np.stack([x[j*PERM_DELAY:j*PERM_DELAY+n] for j in range(PERM_ORDER)],axis=1)
    order=np.argsort(embedded,axis=1,kind='stable')
    counts=np.array([np.all(order==pattern,axis=1).sum() for pattern in PATTERNS])
    assert counts.sum()==n
    probabilities=counts[counts>0]/n
    return float(-np.sum(probabilities*np.log(probabilities))/np.log(len(PATTERNS)))

def compact_features(trials):
    rows=[]
    for trial in trials:
        temporal=resample_poly(sosfiltfilt(TEMPORAL_SOS,trial,axis=-1),32,125,axis=-1)
        assert temporal.shape==(2,1280)
        for start,temporal_start in zip(STARTS,[0,320,640]):
            logs=[];extras=[]
            for ear in range(2):
                f,p=welch(trial[ear,start:start+WINDOW],fs=FS,window='hann',nperseg=1000,
                    noverlap=500,detrend='constant',scaling='density')
                logs.append([np.log(max(integrate_psd(f,p,lo,hi),EPS)) for lo,hi in BANDS.values()])
                x=temporal[ear,temporal_start:temporal_start+640]
                extras.extend([*hjorth(x),spectral_entropy(f,p),permutation_entropy(x)])
            logs=np.asarray(logs)
            ratios=[logs[e,n]-logs[e,2] for e in range(2) for n in [3,1]]
            rows.append(np.r_[logs.ravel(),logs[1]-logs[0],ratios,extras])
    X=np.asarray(rows,float)
    assert X.shape==(len(trials)*3,27) and np.isfinite(X).all()
    return {'A10':X[:,:10].copy(),'B19':X[:,:19].copy(),'C27':X}

# Numerical safeguards and feature-order checks; these do not train classifiers.
zero=compact_features(np.zeros((1,2,5000)))
assert np.isfinite(zero['C27']).all() and np.allclose(zero['C27'][:,10:],0)
t=np.arange(5000)/FS
synthetic=np.stack([np.sin(2*np.pi*10*t),2*np.sin(2*np.pi*10*t)])[None,:,:]
check=compact_features(synthetic)
assert np.array_equal(check['A10'],check['B19'][:,:10])
assert np.array_equal(check['B19'],check['C27'][:,:19])
assert np.allclose(check['B19'][:,12],np.log(4),atol=1e-8) # alpha left/right power ratio
assert permutation_entropy(np.arange(100.))==0
assert np.all((check['C27'][:,[21,22,25,26]]>=-1e-12)&(check['C27'][:,[21,22,25,26]]<=1+1e-12))
print('PASS: dimensions, nested sets, zero signal, log-power difference and entropy ranges.')

feature_maps={name:{} for name in FEATURE_SETS}
for sid in active_ids:
    compact=compact_features(subjects[sid]['trials'])
    for name in ['A10','B19','C27']:feature_maps[name][sid]=compact[name]
    feature_maps['Original80'][sid]=handcrafted_features(subjects[sid]['trials'])
    print('Features ready:',sid,flush=True)
pd.DataFrame([dict(feature_set=tag,index=i,feature=name) for tag,names in FEATURE_SETS.items()
    for i,name in enumerate(names)]).to_csv(OUT/'feature_definitions.csv',index=False)


In [ ]:
def select_features(X,y,method,k):
    scores=np.zeros(X.shape[1]); variable=np.flatnonzero(np.var(X,axis=0)>0)
    if method=='MI': scores[variable]=mutual_info_classif(X[:,variable],y,discrete_features=False,random_state=SEED)
    elif method=='F-test':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore'); scores[variable]=f_classif(X[:,variable],y)[0]
    else: raise ValueError(method)
    scores=np.nan_to_num(scores,nan=0.0,posinf=np.finfo(float).max,neginf=0.0)
    return np.argsort(-scores,kind='stable')[:min(k,X.shape[1])],scores

def balanced_rows(y,seed):
    rng=np.random.default_rng(seed); classes,counts=np.unique(y,return_counts=True); rows=[]
    for c,n in zip(classes,counts):
        idx=np.flatnonzero(y==c); rows.extend(idx.tolist())
        if n<counts.max(): rows.extend(rng.choice(idx,size=counts.max()-n,replace=True).tolist())
    return rng.permutation(rows)

def make_model(name,n_classes):
    if name=='SVM': return SVC(C=1.0,kernel='rbf',gamma='scale')
    if name=='LDA': return LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto')
    if name=='Decision Tree': return DecisionTreeClassifier(max_depth=3,min_samples_leaf=2,random_state=SEED)
    if name=='XGBoost':
        kw=dict(n_estimators=100,max_depth=2,learning_rate=.05,reg_lambda=1.0,subsample=1.0,
                colsample_bytree=1.0,tree_method='hist',n_jobs=1,random_state=SEED,verbosity=0)
        if n_classes>2: kw.update(objective='multi:softprob',num_class=n_classes,eval_metric='mlogloss')
        else: kw.update(objective='binary:logistic',eval_metric='logloss')
        return XGBClassifier(**kw)
    raise ValueError(name)
MODEL_NAMES=['SVM','LDA','Decision Tree','XGBoost']

## Revised methods
All scalers, selectors, correlation pruning and expert fits use training trials only. SVM emits hard votes; the MoE uses LR native probabilities without window-level calibration. Feature selection precedes balancing.

In [ ]:
"""Report-informed extensions. Embedded into the standalone notebooks by build.py."""
from scipy.optimize import minimize
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from threadpoolctl import threadpool_limits
# Tiny matrices are faster and more reproducible with one numerical worker.
_thread_limit=threadpool_limits(limits=1)

def additional_features(trials):
    """Independent-ear positive-gain invariant spectral shape and temporal dynamics."""
    spec, temporal, quality = [], [], []
    for trial in trials:
        filtered = {b: sosfiltfilt(s, trial, axis=-1) for b, s in SOS.items()}
        for start in STARTS:
            shape, dynamics, q = [], [], []
            for ear in range(2):
                x = trial[ear, start:start+WINDOW]
                f, p = welch(x, fs=FS, nperseg=1000, noverlap=500)
                logs = np.log([max(integrate_psd(f,p,*limits), EPS) for limits in BANDS.values()])
                shape.extend(logs-logs.mean())
                for band in ['theta','alpha','beta']:
                    z = filtered[band][ear,start:start+WINDOW].reshape(5,500)
                    lv = np.log(np.maximum(z.var(axis=1),EPS))
                    dynamics.extend([np.polyfit(np.arange(5),lv,1)[0],lv.std()])
                rms = np.sqrt(np.mean(x*x))
                energy = (x.reshape(5,500)**2).sum(axis=1)
                q.extend([np.log(max(rms,EPS)), np.max(np.abs(x))/max(rms,EPS),
                          energy.max()/max(energy.sum(),EPS),
                          integrate_psd(f,p,30,45)/max(integrate_psd(f,p,1,45),EPS)])
            spec.append(shape); temporal.append(dynamics); quality.append(q)
    return np.asarray(spec), np.asarray(temporal), np.asarray(quality)

def fit_predict(X,y,groups,tr,va,learner='LR',selector='All',k=10,neighbors=7,prune=False,balance='weights',seed=42,balance_seed=None):
    assert not set(groups[tr]) & set(groups[va]), 'Trial leakage'
    scaler=StandardScaler().fit(X[tr]); a=scaler.transform(X[tr]); b=scaler.transform(X[va])
    chosen=np.arange(X.shape[1])
    if selector!='All':
        variable=np.flatnonzero(a.var(axis=0)>0); scores=np.zeros(X.shape[1])
        if selector=='MI':
            scores[variable]=mutual_info_classif(a[:,variable],y[tr],n_neighbors=neighbors,random_state=seed)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore'); scores[variable]=f_classif(a[:,variable],y[tr])[0]
        ranked=np.argsort(-np.nan_to_num(scores,nan=0,posinf=1e30),kind='stable')
        chosen=[]
        means=np.array([a[groups[tr]==t].mean(axis=0) for t in np.unique(groups[tr])])
        with np.errstate(invalid='ignore',divide='ignore'): corr=np.nan_to_num(np.corrcoef(means,rowvar=False))
        for j in ranked:
            if not prune or not chosen or np.all(np.abs(corr[j,chosen])<.85): chosen.append(j)
            if len(chosen)==min(k,X.shape[1]): break
        chosen=np.asarray(chosen)
    rows=balanced_rows(y[tr],seed if balance_seed is None else balance_seed) if balance=='oversample' else np.arange(len(tr))
    cw='balanced' if balance=='weights' else None
    if learner=='LR': model=LogisticRegression(C=.1,class_weight=cw,max_iter=2000)
    elif learner=='SVM': model=SVC(C=1,gamma='scale',class_weight=cw)
    elif learner=='LDA': model=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto',priors=np.ones(len(np.unique(y[tr])))/len(np.unique(y[tr])) if cw else None)
    elif learner=='Tree': model=DecisionTreeClassifier(max_depth=3,min_samples_leaf=2,class_weight=cw,random_state=seed)
    elif learner=='XGBoost':
        classes=np.unique(y[tr]); encoder=LabelEncoder().fit(classes)
        model=XGBClassifier(n_estimators=100,max_depth=2,learning_rate=.05,reg_lambda=1.,subsample=1.,colsample_bytree=1.,tree_method='hist',n_jobs=1,random_state=seed,
            objective='multi:softprob' if len(classes)>2 else 'binary:logistic',**({'num_class':len(classes)} if len(classes)>2 else {}))
        weights=np.array([len(tr)/(len(classes)*np.sum(y[tr]==c)) for c in y[tr][rows]]) if cw else None
        model.fit(a[rows][:,chosen],encoder.transform(y[tr][rows]),sample_weight=weights)
        out=np.zeros((len(va),3));out[:,np.searchsorted(LABELS,classes)]=model.predict_proba(b[:,chosen])
        return out,chosen
    else: raise ValueError(learner)
    model.fit(a[rows][:,chosen],y[tr][rows])
    out=np.zeros((len(va),3))
    if learner=='SVM': out=(model.predict(b[:,chosen])[:,None]==LABELS).astype(float)
    else:
        out[:,np.searchsorted(LABELS,model.classes_)]=model.predict_proba(b[:,chosen])
    assert np.isfinite(out).all() and np.allclose(out.sum(axis=1),1)
    return out,chosen

def gate_fit(P,Q,y,penalty=1.):
    """Class-balanced trial-level Brier objective, shrunk toward equal routing."""
    target=(y[:,None]==LABELS).astype(float)
    sw=np.array([1/(len(np.unique(y))*np.sum(y==v)) for v in y])
    def objective(flat):
        w=softmax(Q@flat.reshape(Q.shape[1],P.shape[1]),axis=1)
        prediction=np.einsum('ne,nec->nc',w,P)
        # Sum keeps the same per-coefficient shrinkage as gate dimensions change.
        return np.sum(sw*np.sum((prediction-target)**2,axis=1))+penalty*np.sum(flat**2)
    result=minimize(objective,np.zeros(Q.shape[1]*P.shape[1]),method='L-BFGS-B')
    if not result.success: return np.zeros((Q.shape[1],P.shape[1])),str(result.message)
    return result.x.reshape(Q.shape[1],P.shape[1]),'ok'

def evaluate_prediction(rows,predictions,sid,fold,method,va,p,seed,policy):
    d=subjects[sid]; truth=d['segment_y'][va]; tids=d['trial_ids'][va]
    guessed=LABELS[p.argmax(axis=1)]
    for population,mask in [('all',np.ones(len(va),bool)),('common',np.array([(sid,int(t)) not in SUSPECTED_TRIALS for t in tids]))]:
        for level in ['window','trial']:
            yy=truth[mask]; pp=p[mask]
            if level=='trial':
                ts=np.unique(tids[mask]); yy=np.array([truth[tids==t][0] for t in ts]); pp=np.array([p[tids==t].mean(axis=0) for t in ts])
            yh=LABELS[pp.argmax(axis=1)]
            rows.append(dict(subject=sid,fold=fold,seed=seed,policy=policy,method=method,population=population,level=level,
                macro_f1=f1_score(yy,yh,labels=LABELS,average='macro',zero_division=0),accuracy=accuracy_score(yy,yh),n=len(yy),
                **{f'f1_{c}':float(v) for c,v in zip(LABELS,f1_score(yy,yh,labels=LABELS,average=None,zero_division=0))}))
    for i,idx in enumerate(va):
        predictions.append(dict(subject=sid,fold=fold,seed=seed,policy=policy,method=method,trial=int(tids[i]),window=int(idx%3),truth=int(truth[i]),prediction=int(guessed[i]),**{f'p_{c}':float(p[i,j]) for j,c in enumerate(LABELS)}))

def run_revision(mode='moe',seeds=(42,),policies=('include',)):
    scores=[]; predictions=[]; routing=[]; selection=[]
    config.update(balance='per-configuration: weights or random oversampling',
        model_selection='inner trial-OOF gate learning; fixed expert hyperparameters; exploratory outer comparisons',
        primary_evaluation='all original held-out windows',secondary_evaluation='common nonsuspected windows; trial-pooled predictions',
        lr=dict(C=.1,class_weight='balanced',max_iter=2000),quality_clip=5,gate_penalty=1,
        new_feature_names=dict(Shape10=[f'{ear}_{band}_centered_log_power' for ear in EARS for band in BANDS],
            Temporal12=[f'{ear}_{band}_logvariance_{stat}' for ear in EARS for band in ['theta','alpha','beta'] for stat in ['slope','std']]),
        quality_names=[f'{ear}_{q}' for ear in EARS for q in ['log_rms','crest','max_1s_energy_fraction','gamma_fraction']])
    for sid in active_ids:
        d=subjects[sid]; spectral,temporal,quality=additional_features(d['trials'])
        maps={k:v[sid] for k,v in feature_maps.items()};maps.update(Shape10=spectral,Temporal12=temporal,Invariant22=np.c_[spectral,temporal])
        y=d['segment_y']; groups=d['trial_ids']
        for seed in seeds:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore'); splits=list(StratifiedKFold(3,shuffle=True,random_state=seed).split(d['y'],d['y']))
            for fold,(train_trials,val_trials) in enumerate(splits,1):
                if seed==42:
                    frozen=FROZEN_FOLD_IDS[sid][fold-1]
                    assert set(train_trials+1)==set(frozen['train_trials']) and set(val_trials+1)==set(frozen['validation_trials'])
                va=np.flatnonzero(np.isin(groups,val_trials+1))
                for policy in policies:
                    tt=np.array([t for t in train_trials+1 if policy=='include' or (sid,int(t)) not in SUSPECTED_TRIALS])
                    tr=np.flatnonzero(np.isin(groups,tt)); assert not set(groups[tr])&set(groups[va])
                    if mode=='compact':
                        configs=[('Original80','F',20,3,False,'oversample'),('Original80','F',20,3,False,'weights'),
                                 ('Original80','MI',20,3,False,'weights'),('Original80','MI',20,7,False,'weights'),
                                 ('Original80','MI',10,7,False,'weights'),('Original80','MI',10,7,True,'weights')]
                        configs += [(name,'All',10,7,False,'weights') for name in ['A10','B19','C27','Shape10','Temporal12','Invariant22']]
                        for name,selector,k,nn,prune,balance in configs:
                            for learner in ['SVM','LDA','Tree','LR','XGBoost']:
                                tag=f'{name}/{selector}{k if selector!="All" else ""}/nn{nn}/prune{prune}/{balance}/{learner}'
                                p,chosen=fit_predict(maps[name],y,groups,tr,va,learner,selector,k,nn,prune,balance,seed,balance_seed=seed+sid*10+fold)
                                evaluate_prediction(scores,predictions,sid,fold,tag,va,p,seed,policy)
                                selection.append(dict(subject=sid,fold=fold,seed=seed,policy=policy,method=tag,selected=json.dumps(chosen.tolist())))
                    else:
                        experts=['A10','Shape10','Temporal12']; P=[]
                        for name in experts:
                            p,_=fit_predict(maps[name],y,groups,tr,va,seed=seed);P.append(p)
                            evaluate_prediction(scores,predictions,sid,fold,name,va,p,seed,policy)
                        P=np.stack(P,axis=1)
                        evaluate_prediction(scores,predictions,sid,fold,'Equal',va,P.mean(axis=1),seed,policy)
                        for leave in range(3): evaluate_prediction(scores,predictions,sid,fold,'Without_'+experts[leave],va,np.delete(P,leave,axis=1).mean(axis=1),seed,policy)
                        yt=d['y'][tt-1]; eligible=all(np.sum(yt==c)>=2 for c in LABELS)
                        oof=np.zeros((len(tt),3,3)); oq=np.zeros((len(tt),8)); reason='insufficient class trials'
                        outer_scaler=StandardScaler().fit(quality[tr].reshape(len(tt),3,8).mean(axis=1))
                        qval=outer_scaler.transform(quality[va].reshape(len(val_trials),3,8).mean(axis=1))
                        qval=np.repeat(np.clip(qval,-5,5),3,axis=0)
                        if eligible:
                            inner=StratifiedKFold(2,shuffle=True,random_state=seed+sid*10+fold)
                            for it,iv in inner.split(tt,yt):
                                ti=np.flatnonzero(np.isin(groups,tt[it])); vi=np.flatnonzero(np.isin(groups,tt[iv]))
                                # vi follows original trial order; tt and iv are sorted.
                                for e,name in enumerate(experts):
                                    ip,_=fit_predict(maps[name],y,groups,ti,vi,seed=seed)
                                    oof[iv,e]=ip.reshape(len(iv),3,3).mean(axis=1)
                                qs=StandardScaler().fit(quality[ti].reshape(len(it),3,8).mean(axis=1))
                                oq[iv]=np.clip(qs.transform(quality[vi].reshape(len(iv),3,8).mean(axis=1)),-5,5)
                            reason='ok'
                        for gate in ['Constant','Quality','Quality_no_amplitude','Shuffled_quality']:
                            qtrain=oq.copy(); qtest=qval.copy()
                            if gate=='Constant': qtrain=qtrain[:,:0];qtest=qtest[:,:0]
                            if gate=='Quality_no_amplitude': qtrain=np.delete(qtrain,[0,4],axis=1);qtest=np.delete(qtest,[0,4],axis=1)
                            if gate=='Shuffled_quality': qtrain=qtrain[np.random.default_rng(seed+sid+fold).permutation(len(tt))]
                            qa=np.c_[np.ones(len(tt)),qtrain]; qb=np.c_[np.ones(len(va)),qtest]
                            coef,status=gate_fit(oof,qa,yt) if eligible else (np.zeros((qa.shape[1],3)),reason)
                            weights=softmax(qb@coef,axis=1)
                            evaluate_prediction(scores,predictions,sid,fold,gate,va,np.einsum('ne,nec->nc',weights,P),seed,policy)
                            for i,idx in enumerate(va): routing.append(dict(subject=sid,fold=fold,seed=seed,policy=policy,method=gate,trial=int(groups[idx]),window=int(idx%3),status=status,**{name:float(weights[i,e]) for e,name in enumerate(experts)}))
        print('Completed',mode,'subject',sid,flush=True)
    result=pd.DataFrame(scores); result.to_csv(OUT/f'{mode}_fold_scores.csv',index=False)
    pd.DataFrame(predictions).to_csv(OUT/f'{mode}_predictions.csv',index=False)
    pd.DataFrame(routing).to_csv(OUT/f'{mode}_routing.csv',index=False)
    pd.DataFrame(selection).to_csv(OUT/f'{mode}_selected_features.csv',index=False)
    keys=['method','policy','population','level']
    subject=result.groupby(keys+['subject'])[['macro_f1','accuracy']].mean().reset_index()
    subject.to_csv(OUT/f'{mode}_subject_scores.csv',index=False)
    summary=subject.groupby(keys)[['macro_f1','accuracy']].mean().reset_index()
    summary.to_csv(OUT/f'{mode}_summary.csv',index=False)
    (OUT/f'{mode}_config.json').write_text(json.dumps(dict(config,seeds=list(seeds),policies=list(policies),mode=mode,
        primary='all validation windows; fold then seed then subject mean',gate='inner OOF trial-level class-balanced Brier; penalty 1; quality clip 5',
        tie_break='first label [-1,0,1]',exploratory=True),indent=2))
    return summary


In [ ]:
SEEDS=[42]
POLICIES=['include','exclude']
summary=run_revision('compact',SEEDS,POLICIES)
display(summary[(summary.population=='all') & (summary.level=='window')].sort_values('macro_f1',ascending=False))